In [293]:
import yaml
from tempfile import mkdtemp
from os.path import join, basename
from jinja2 import Template
from tqdm import tqdm

#fp_workdir = mkdtemp()

# this list of qiime2 plugins to be NOT installed is a result of:
# 1) create a full qiime2 environment with installed qp-qiime2
# 2) ipython
#    a) the same functions as in qp-qiime2/__init__.py
#    b) modify function qp-qiime2/util.py::register_qiime2_commands in a way that it stores plugins+functions in a dict, which
#       would have been registered via: plugin.register_command(qiime_cmd), line 328
#    c) subtract these white-listed plugins from what is listed if "qiime" is executed
blacklist = [
    'alignment',
    #'composition',
    'cutadapt',
    'dada2',
    'deblur',
    'demux',
    'fragment-insertion',
    'gneiss',
    'greengenes2',
    'metadata',
    'quality-control',
    'quality-filter',
    'sourcetracker2',
    'vsearch',
]

# dependencies most likely needed, but flagges by this naive mechanism as superfluous.
# Note that we do NOT follow dependencies of dependencies of plugins!
whitelist = [
    'click',
    'decorator',
    'yaml',
    'pyyaml',
    'ca-certificates',
    'certifi',
    'openssl'
    'libgcc',   
    'libgfortran',
    'libllvm15',  
    'tomlkit',
    'formulaic',
    'typing-extensions',
    'interface_meta',
    'wrapt',  
    'altair',
    'narwhals',
    'attrs',
    'zipp',
    'importlib-metadata',
    'rpds-py',
    'referencing',
    'jsonschema-specifications',
    'jsonschema',
    'nose',
]

release = "2023.5"  # this is the used Qiime2 release

# download full dependency list of official qiime2 release
fp_conda = join(fp_workdir, "qiime2-%s-py38-linux-conda.yml" % release)
cmd = 'if [ ! -f %s ]; then export https_proxy="http://proxy.computational.bio.uni-giessen.de:3128"; wget "https://data.qiime2.org/distro/core/%s" -O %s; fi' % (fp_conda, basename(fp_conda), fp_conda)
!$cmd
with open(fp_conda, "r") as file:
    qiime2 = yaml.safe_load(file)

# iterate individual qiime2 plugins and retriev their individual dependency lists
plugin_dependencies = dict()
qiime_plugins = [ d.split('=')[0] for d in qiime2['dependencies'] if d.startswith('q2-')]
for plugin in tqdm(qiime_plugins):
    fp_conda_plugin = join(fp_workdir, "%s-%s.0.yaml" % (plugin, qrelease))
    cmd = 'if [ ! -f %s ]; then export https_proxy="http://proxy.computational.bio.uni-giessen.de:3128"; wget "https://raw.githubusercontent.com/qiime2/%s/refs/tags/%s.0/ci/recipe/meta.yaml" -O - | grep -v "^{%%" > %s; fi' % (fp_conda_plugin, plugin, release, fp_conda_plugin)
    #print(cmd)
    !$cmd

    with open(fp_conda_plugin) as f:
        template = Template(f.read())
        rendered = template.render(
            version="1.2.3",
            python="3.9",
            scikit_bio="0.5.8",
            qiime2_epoch="2025",
            qiime2="2025.1",
            q2_types="2025.1",
        )
        plugin_yaml = yaml.safe_load(rendered)
        plugin_dependencies[plugin] = {package.split()[0] for _, packages in plugin_yaml['requirements'].items() for package in packages}

100%|██████████| 23/23 [00:04<00:00,  4.81it/s]


In [294]:
def read_conda_out(fp):
    packages = set()
    with open(fp) as f:
        for line in f.readlines():
            if line.startswith('  + '):
                plus, package, version, channel, _, _ = line.split()
                assert plus == '+'
                packages |= set([package])
    return packages

nonq2dependencies = {d for ds in plugin_dependencies.values() for d in ds if not d.startswith('q2-') and d not in ['qiime2', 'q2templates']}
channels = ' '.join(list(map(lambda x: '-c %s' % x, [c for c in qiime2['channels'] if not c.startswith('qiime2')])))
full_dependency_list = dict()
for dependency in tqdm(sorted(list(nonq2dependencies))):
    fp_conda_dependency = join(fp_workdir, "%s.out" % dependency)
    cmd = ('if [ ! -f %s ]; '
           'then export https_proxy="http://proxy.computational.bio.uni-giessen.de:3128"; '
                 '/homes/sjanssen/miniconda3/bin/mamba create --name fake %s %s --dry-run -vvv > %s 2> %s; '
           'fi' % (fp_conda_dependency, dependency, channels, fp_conda_dependency, fp_conda_dependency.replace('.out', '.err')))
    #print(cmd)
    !$cmd

    full_dependency_list[dependency] = read_conda_out(fp_conda_dependency)
    #break

100%|██████████| 55/55 [00:10<00:00,  5.22it/s]


In [295]:
# check which dependencies are used from black-listed plugins, BUT not in plugins used in qiita
deps = {'black': set(), 'white': set()}
for plugin in plugin_dependencies.keys():
    color = 'white'    
    if plugin.split('q2-')[-1] in blacklist:
        color = 'black'
    deps[color] |= set(plugin_dependencies[plugin])

deps_full = dict()
for color in deps.keys():
    deps_full[color] = set()
    for level1_dependency in deps[color]:
        # print(color, level1_dependency, full_dependency_list[level1_dependency])
        if level1_dependency in full_dependency_list.keys():
            deps_full[color] |= full_dependency_list[level1_dependency]
        deps_full[color] |= set([level1_dependency])

In [296]:
spare_dependency_candidates = (deps_full['black'] - deps_full['white']) - set(whitelist)

In [297]:
# list of packages, which cannot be removed, as they are not present in qiime env
missing = set([
    'openmpi',
    'narwhals',
    'dnspython',
    'pymongo',
    'r-multcomp',
    'clustalw',
    'jsonschema-specifications',
    'mpi',
    'bioconductor-ucsc.utils',
    'prank',
    'libgettextpo',
    'r-gtools',
    'pasta',
    'r-zoo',
    'tbb-devel',
    'r-reformulas',
    'python-zlib-ng',
    'rpds-py',
    'bioconductor-s4arrays',
    'q2-sourcetracker2',
    'pplacer',
    'r-s7',
    'bioconductor-sparsearray',
    'libasprintf',
    'r-abind',
    'r-th.data',
    'referencing',
    'r-sandwich',
    'muscle',
    'bioconductor-pwalign',
    'q2-greengenes2',
])
# 
addremove = set([
    'blast'  # as 
])
#missing = set()
'conda remove -n qiime2 --force %s' % ' '.join(list((spare_dependency_candidates | set(map(lambda x: 'q2-%s' % x, blacklist))) - missing | addremove))

'conda remove -n qiime2 --force htslib r-bh pigz bioconductor-summarizedexperiment q2-demux python-isal q2-dada2 q2-alignment gneiss r-futile.logger r-rcppparallel q2-fragment-insertion dnaio mafft pbzip2 cutadapt q2-quality-filter q2-quality-control gawk pcre xopen q2-vsearch q2-deblur r-futile.options r-hwriter sniffio xyzservices sortmerna bioconductor-biocparallel q2-metadata bioconductor-genomicalignments hmmer bioconductor-decontam bioconductor-rhtslib bioconductor-delayedarray bioconductor-shortread bioconductor-dada2 deblur samtools r-matrixstats r-lambda.r r-bitops isa-l bowtie2 sepp r-snow bioconductor-genomicranges openjdk q2-cutadapt bioconductor-matrixgenerics r-formatr q2-gneiss blast bokeh dendropy giflib bioconductor-rsamtools'